# Tópicos Avançados em IA — Aula 04
## Aprendizado ativo: qual ponto eu rotulo agora?

Prof. James Moraes de Almeida

Roteiro:
1. O enigma dos aços.
2. O laço do aprendizado ativo, escrito à mão, num problema 2D: duas luas e um SVM.
3. Estratégias de incerteza nos dígitos: menor confiança, margem, entropia.
4. O comitê — e o mesmo laço pronto, no `scikit-activeml`.
5. Regressão ativa: escolher o próximo aço a medir.
6. Quando atrapalha, para ir além — e a **Atividade AL** (no fim).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from contourpy import contour_generator
from matplotlib.ticker import FuncFormatter
from sklearn.datasets import load_digits, make_moons
from sklearn.ensemble import BaggingClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.figsize"] = (10, 4)

# Cores e traços seguros para daltônicos: nenhuma curva se distingue só pela cor — cada uma tem um
# traço próprio e o nome escrito na ponta. As duas classes são azul/laranja com formas diferentes.
AZUL, LARANJA, CINZA, TINTA = "#0072B2", "#E69F00", "#BBBBBB", "#1D5264"
ESTILO = {
    "aleatória":             dict(color="#7A7A7A", ls="--", lw=2.2),
    "incerteza":             dict(color=TINTA, ls="-", lw=2.6),
    "margem":                dict(color=TINTA, ls="-", lw=2.6),
    "menor confiança":       dict(color=AZUL, ls=":", lw=2.8),
    "entropia":              dict(color="#D55E00", ls="-.", lw=2.2),
    "comitê":                dict(color=LARANJA, ls="-", lw=2.4, marker="o", markevery=10, ms=6),
    "desacordo das árvores": dict(color=LARANJA, ls="-", lw=2.4, marker="o", markevery=5, ms=6),
}
VIRGULA = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))
COR_NOME = {LARANJA: "#9A5F00"}                     # o nome em laranja escuro: o claro some no fundo branco

def br(v, casas=3):
    return f"{v:.{casas}f}".replace(".", ",")

def curvas_com_nome(ax, x, curvas, faixa=("aleatória",), folga_x=0.34):
    """A média de cada estratégia com o nome escrito na ponta da curva (no lugar da legenda) e a
    faixa ± desvio padrão só das estratégias em `faixa`."""
    pontas = []
    for nome, A in curvas.items():
        m, d = A.mean(0), A.std(0)
        ax.plot(x, m, **ESTILO[nome])
        if nome in faixa:
            ax.fill_between(x, m - d, m + d, color=ESTILO[nome]["color"], alpha=0.15, lw=0)
        pontas.append([m[-1], nome])
    lo, hi = ax.get_ylim()
    folga = 0.065 * (hi - lo)
    pontas.sort()                                   # afasta os nomes que cairiam um sobre o outro…
    for k in range(1, len(pontas)):
        pontas[k][0] = max(pontas[k][0], pontas[k - 1][0] + folga)
    excesso = pontas[-1][0] - (hi - folga / 2)      # …e, se o de cima passar do topo, desce todos
    if excesso > 0:
        for pt in pontas:
            pt[0] -= excesso
    for y, nome in pontas:
        ax.annotate(nome, (x[-1], y), xytext=(8, 0), textcoords="offset points", va="center",
                    color=COR_NOME.get(ESTILO[nome]["color"], ESTILO[nome]["color"]), fontweight="bold",
                    annotation_clip=False)
    ax.set_ylim(lo, hi)
    ax.set_xlim(x[0], x[-1] + folga_x * (x[-1] - x[0]))
    ax.set_xticks([t for t in ax.get_xticks() if x[0] <= t <= x[-1]])
    ax.yaxis.set_major_formatter(VIRGULA)
    ax.grid(alpha=0.3)

## 1. O enigma

Temos **312 aços** com composição conhecida (13 elementos, em % de massa). Medir o
limite de escoamento de cada um custa **uma semana de laboratório**.

> Quantos aços precisamos medir, em média, para encontrar o mais resistente?

**Escreva o seu palpite antes de seguir.** A resposta sai na Aula 06, por dois caminhos.

In [ ]:
acos = pd.read_csv("data/acos.csv")
print(acos.shape)
fig, ax = plt.subplots(figsize=(6, 4.8))
acos["limite_escoamento_MPa"].plot.hist(bins=30, ax=ax, color=AZUL, edgecolor="white")
ax.set_title("Limite de escoamento dos 312 aços", fontsize=16)
ax.set_xlabel("limite de escoamento (MPa)", fontsize=15); ax.set_ylabel("número de aços", fontsize=15)
ax.tick_params(labelsize=13);
acos.head()

## 2. O laço do aprendizado ativo

### O problema, formalmente

Temos um conjunto **rotulado** pequeno $\mathcal{L} = \{(x_i, y_i)\}$, um conjunto **não rotulado**
grande $\mathcal{U} = \{x_j\}$ (o *pool*) e um **oráculo** que devolve o rótulo $y$ de qualquer
$x \in \mathcal{U}$ — a um custo. Com um orçamento de $B$ rótulos, queremos o melhor modelo possível.

A cada passo, uma **função de aquisição** $a(x)$ dá uma nota a cada candidato, e rotulamos

$$x^* = \arg\max_{x \in \mathcal{U}} a(x).$$

A escolha aleatória é o caso $a(x)$ = constante. Settles (2009) distingue três cenários: o de hoje
(*pool-based*, escolher de um conjunto dado), o de fluxo (cada ponto chega e se decide na hora se vale
rotular) e o de síntese (o modelo **inventa** o ponto a consultar — como propor uma composição de aço
que não está na tabela).

```
rótulos iniciais (poucos) → treina o modelo → mede a incerteza em cada ponto sem rótulo
        ↑                                                     ↓
   retreina  ←  o oráculo (laboratório, simulação, anotador) rotula o mais incerto
```

Tudo depende da **função de aquisição**: a regra que diz qual ponto vale mais a pena
rotular. Começamos com a mais simples, num problema 2D em que dá para ver.

### O exemplo: duas luas e um SVM

**Os dados.** 400 pontos sintéticos de **duas classes**, em forma de duas luas encaixadas, com um pouco de
ruído (`make_moons`). A classe de cada ponto está guardada; o laço só a vê quando pede o rótulo ao
oráculo — aqui, simulado.

**O modelo.** Um SVM (*máquina de vetores de suporte*) com kernel RBF: um classificador que traça uma
**fronteira curva** entre as classes — de um lado ele diz azul, do outro, laranja. A geometria dele vem logo
depois do resultado.

**As estratégias.** A **incerteza** rotula o ponto ainda sem rótulo mais perto da fronteira atual; a
**aleatória** sorteia. As duas partem dos mesmos 4 rótulos iniciais (com as duas classes) e ganham mais 20.

In [ ]:
X2, y2 = make_moons(n_samples=400, noise=0.2, random_state=3)
XLIM, YLIM = (-1.6, 2.6), (-1.15, 1.75)        # a janela dos gráficos: os dados vão de −1,4 a 2,4 e de −0,9 a 1,5

def laco_2d(estrategia, n_consultas=20, semente=0, erro=None):
    """estrategia: "aleatória", "incerteza" (o ponto mais perto da fronteira do SVM) ou "probabilidade"
    (a probabilidade calibrada do SVM mais perto de 0,5). `erro`: a chance de o oráculo errar o rótulo de
    cada ponto (seção 6); None = um oráculo que nunca erra. Devolve o modelo final, os índices rotulados e
    os rótulos que o oráculo deu."""
    rng = np.random.default_rng(semente)
    rng_oraculo = np.random.default_rng(10_000 + semente)   # os erros do oráculo não mexem nos sorteios
    while True:                                  # os 4 iniciais precisam ter as duas classes
        rotulados = list(rng.choice(len(y2), 4, replace=False))
        if len(set(y2[rotulados])) == 2:
            break
    y_dado = y2.copy()                           # o que o oráculo respondeu (= y2, se ele nunca erra)
    prob = estrategia == "probabilidade"
    novo = lambda: SVC(gamma=1.0, probability=prob, random_state=0)
    for _ in range(n_consultas):
        modelo = novo().fit(X2[rotulados], y_dado[rotulados])
        pool = np.setdiff1d(np.arange(len(y2)), rotulados)
        if estrategia == "aleatória":
            q = rng.choice(pool)
        elif estrategia == "incerteza":
            q = pool[np.argmin(np.abs(modelo.decision_function(X2[pool])))]
        else:
            q = pool[np.argmin(np.abs(modelo.predict_proba(X2[pool])[:, 1] - 0.5))]
        if erro is not None and rng_oraculo.uniform() < erro[q]:
            y_dado[q] = 1 - y2[q]                # o oráculo errou este
        rotulados.append(q)
    return novo().fit(X2[rotulados], y_dado[rotulados]), rotulados, y_dado

GX, GY = np.meshgrid(np.linspace(*XLIM, 300), np.linspace(*YLIM, 220))

def fronteira(modelo, minimo=1.0):
    """Os trechos da curva f(x) = 0 dentro da janela. Trechos com menos de `minimo` de comprimento (ilhas
    minúsculas da fronteira, nas bordas) são descartados: só poluem a figura."""
    zz = modelo.decision_function(np.c_[GX.ravel(), GY.ravel()]).reshape(GX.shape)
    trechos = contour_generator(GX, GY, zz).lines(0.0)
    return [t for t in trechos if np.linalg.norm(np.diff(t, axis=0), axis=1).sum() >= minimo]

def desenhar_luas(ax, modelo, rotulados, titulo):
    ax.set_aspect("equal")
    sem = np.setdiff1d(np.arange(len(y2)), rotulados)
    ax.scatter(*X2[sem].T, s=10, color=CINZA, label="ainda sem rótulo")
    for c, cor, marca, nome in [(0, AZUL, "o", "rotulado: classe azul"), (1, LARANJA, "^", "rotulado: classe laranja")]:
        r = [i for i in rotulados if y2[i] == c]
        ax.scatter(*X2[r].T, s=110, color=cor, marker=marca, edgecolor="#1A1A1A", lw=0.8, zorder=3, label=nome)
    for k, t in enumerate(fronteira(modelo)):
        ax.plot(*t.T, color="#1A1A1A", lw=2.2, label="fronteira do modelo" if k == 0 else None)
    ax.set_xlim(XLIM); ax.set_ylim(YLIM); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(titulo)

fig, eixos = plt.subplots(1, 2, figsize=(13, 5.0))
for ax, estr, nome in zip(eixos, ["aleatória", "incerteza"], ["Aleatória", "Incerteza"]):
    modelo, rot, _ = laco_2d(estr)
    desenhar_luas(ax, modelo, rot, f"{nome}: 4 + 20 rótulos — acurácia {br(modelo.score(X2, y2))}")
fig.legend(*eixos[0].get_legend_handles_labels(), loc="lower center", ncol=4, frameon=False)
fig.tight_layout(rect=(0, 0.07, 1, 1))

**Repare onde caem os rótulos.** A incerteza gasta os rótulos ao longo da fronteira — a região que decide a
classificação; a aleatória espalha boa parte deles longe dela, onde o modelo já acertava. A acurácia é
medida nos 400 pontos.

### Foi sorte? Repetindo 20 vezes

Uma execução só é uma anedota: outro sorteio dos 4 rótulos iniciais (e dos pontos da aleatória) dá outro
resultado. Cada sorteio diferente é uma **semente** (o `semente=s` abaixo). Repetimos tudo com 20 sementes
e contamos em quantas a incerteza termina com acurácia maior que a aleatória — com 14 rótulos (4 + 10) e
com 24 (4 + 20).

In [ ]:
vinte = {}
for n in (10, 20):
    acc = {e: np.array([laco_2d(e, n, s)[0].score(X2, y2) for s in range(20)])
           for e in ["aleatória", "incerteza", "probabilidade"]}
    vinte[n] = acc
    print(f"--- {4 + n} rótulos (4 iniciais + {n} escolhidos)")
    for e, a in acc.items():
        extra = "" if e == "aleatória" else f"   ganha da aleatória em {np.sum(a > acc['aleatória'])} de 20 repetições"
        print(f"  {e:14s} {a.mean():.3f} ± {a.std():.3f}{extra}")

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True)
for ax, n in zip(eixos, (10, 20)):
    a, b = vinte[n]["aleatória"], vinte[n]["incerteza"]
    for va, vb in zip(a, b):                    # cada linha é uma repetição: sobe se a incerteza ganhou
        ganhou = vb > va
        ax.plot([0, 1], [va, vb], color=TINTA if ganhou else LARANJA, ls="-" if ganhou else "--",
                lw=1.3, alpha=0.75, marker="o", ms=4)
    ax.plot([0, 1], [a.mean(), b.mean()], color="black", lw=4, marker="D", ms=9, zorder=3, label="média")
    ax.set_xticks([0, 1], ["aleatória", "incerteza"]); ax.set_xlim(-0.3, 1.3)
    ax.set_title(f"{4 + n} rótulos: a incerteza ganhou em {np.sum(b > a)} de 20")
    ax.yaxis.set_major_formatter(VIRGULA); ax.grid(axis="y", alpha=0.3)
eixos[0].set_ylabel("acurácia (cada linha = uma repetição)")
eixos[1].legend(loc="lower right", frameon=False)
fig.tight_layout()

**Para ir além: e se a incerteza for medida pela probabilidade?** A linha "probabilidade" da tabela usa a
mesma ideia medida de outro jeito: a probabilidade do SVM (`predict_proba`) mais perto de 0,5. Com 14
rótulos ela **perde** da aleatória e só se recupera com 24. O SVM não dá probabilidade por natureza: o
scikit-learn a obtém ajustando uma sigmoide por cima, com validação cruzada interna em poucos pontos (o
escalonamento de Platt, abaixo) — e com poucos rótulos essa calibração é justamente o que o modelo não tem
como fazer bem. **A medida da incerteza importa tanto quanto a ideia.**

### Por que a distância à fronteira funciona: a geometria do SVM

O SVM classifica pelo sinal da **função de decisão**
$f(x) = \sum_i \alpha_i\, y_i\, K(x_i, x) + b$, em que $K$ é o kernel (aqui, o RBF da Aula 05) e os
$x_i$ com $\alpha_i > 0$ são os vetores de suporte. A fronteira é $f(x) = 0$, e $|f(x)|$ é proporcional à
distância de $x$ até ela no espaço do kernel. `decision_function` devolve $f(x)$.

Tong e Koller (2001) deram a justificativa: o conjunto de fronteiras compatíveis com os rótulos já
vistos (o *espaço de versões*) encolhe mais depressa quando se rotula o ponto mais perto da fronteira
atual — cada rótulo corta esse espaço mais ou menos ao meio. É a mesma ideia de uma busca binária.

A probabilidade do `predict_proba` é outra coisa: uma sigmoide $1/(1 + e^{A f(x) + B})$ ajustada por
validação cruzada interna de 5 dobras (o escalonamento de Platt). Com 4 a 14 rótulos, cada dobra tem
um ou dois pontos, e os $A$, $B$ estimados são ruído.

## 3. Estratégias de incerteza — dígitos

**O problema.** 1797 imagens de 8×8 pixels de dígitos escritos à mão: **10 classes**. Rotular uma imagem
(alguém olhar e dizer que dígito é) é o custo. **O modelo:** uma regressão logística, que dá a cada imagem
uma **probabilidade para cada um dos 10 dígitos**.

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd = Xd / 16
fig, eixos = plt.subplots(2, 10, figsize=(11, 2.9))
for d in range(10):
    for lin, i in enumerate(np.flatnonzero(yd == d)[:2]):
        ax = eixos[lin, d]
        ax.imshow(Xd[i].reshape(8, 8), cmap="gray_r"); ax.set_xticks([]); ax.set_yticks([])
        if lin == 0:
            ax.set_title(str(d))
fig.suptitle(f"{len(yd)} imagens 8×8 — dois exemplos de cada dígito", y=1.02);

**O que o modelo diz de uma imagem.** Treinamos a regressão logística com 30 imagens rotuladas ao acaso (os 10
dígitos aparecem entre elas). Para cada imagem ainda sem rótulo, ela devolve **10 probabilidades**, uma para cada
dígito, que somam 1. Em ordem decrescente, a maior se chama **p₁** — o palpite favorito do modelo — e a segunda,
**p₂**. A resposta certa o modelo **não** sabe: só o oráculo, quando se pede o rótulo.

In [ ]:
def utilidade(P, estrategia):
    Ps = np.sort(P, axis=1)
    if estrategia == "menor confiança":
        return 1 - Ps[:, -1]
    if estrategia == "margem":
        return -(Ps[:, -1] - Ps[:, -2])
    if estrategia == "entropia":
        return -(P * np.log(P + 1e-12)).sum(axis=1)
    raise ValueError(estrategia)

Xp, Xt, yp, yt = train_test_split(Xd, yd, test_size=0.3, random_state=0, stratify=yd)
rot = list(np.random.default_rng(4).choice(len(yp), 30, replace=False))   # 30 ao acaso, com os 10 dígitos
modelo = LogisticRegression(max_iter=2000).fit(Xp[rot], yp[rot])
assert len(modelo.classes_) == 10
pool = np.setdiff1d(np.arange(len(yp)), rot)
P = modelo.predict_proba(Xp[pool])                 # 10 probabilidades para cada imagem do pool

def barras(ax, p, marcar=True):
    """As 10 probabilidades de uma imagem; p₁ cheia e p₂ hachurada (marcar=False: todas iguais)."""
    o = np.argsort(-p)
    b = ax.bar(modelo.classes_, p, color=CINZA, edgecolor="#555555")
    if marcar:
        b[o[0]].set_color(TINTA)
        b[o[1]].set_facecolor("white"); b[o[1]].set_edgecolor(TINTA); b[o[1]].set_hatch("///")
        ax.text(o[0], p[o[0]] + 0.015, "p₁", ha="center", va="bottom", fontweight="bold")
        ax.text(o[1], p[o[1]] + 0.015, "p₂", ha="center", va="bottom", fontweight="bold")
    ax.set_xticks(range(10)); ax.set_xlabel("dígito")
    ax.yaxis.set_major_formatter(VIRGULA)
    return o

Duas imagens do pool: aquela em que o modelo está **mais seguro** e aquela em que está **mais em dúvida**.

In [ ]:
seguro, duvida = int(np.argmax(P.max(1))), int(np.argmin(P.max(1)))
fig, eixos = plt.subplots(1, 4, figsize=(13, 3.6), gridspec_kw={"width_ratios": [0.62, 1.5, 0.62, 1.5]})
for j, (k, nome) in enumerate([(seguro, "modelo seguro"), (duvida, "modelo em dúvida")]):
    p, i = P[k], pool[k]
    ax = eixos[2 * j]
    ax.imshow(Xp[i].reshape(8, 8), cmap="gray_r"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(f"o oráculo revela:\né um {yp[i]}", fontsize=13)
    ax = eixos[2 * j + 1]
    o = barras(ax, p)
    ax.set_ylim(0, 1.08); ax.set_ylabel("probabilidade")
    ax.set_title(f"{nome}: palpite {modelo.classes_[o[0]]}, p₁ = {br(p[o[0]], 2)}", fontweight="bold")
    print(f"{nome}: o palpite é {modelo.classes_[o[0]]} com p₁ = {p[o[0]]:.2f}, p₂ = {p[o[1]]:.2f} "
          f"(o {modelo.classes_[o[1]]}) — é um {yp[i]}")
fig.tight_layout(w_pad=1.2)

Rotular a imagem da esquerda ensina pouco: o modelo já acerta, e com folga. A da direita ensina mais: o palpite
era fraco — e errado. **Toda estratégia de incerteza procura imagens como a da direita.** Com mais de duas classes,
"em dúvida" tem mais de uma definição; cada estratégia faz uma pergunta diferente às mesmas 10 probabilidades:

| estratégia | a pergunta | escolhe o ponto com |
|---|---|---|
| menor confiança | o palpite favorito é fraco? | maior `1 − p₁` |
| margem | os dois primeiros estão empatados? | menor `p₁ − p₂` |
| entropia | a dúvida está espalhada por todas as classes? | maior `−Σ pᵢ ln pᵢ` |

Em fórmulas, com $p_{(1)} \ge p_{(2)} \ge \dots$ as probabilidades ordenadas de um ponto $x$:

$$a_{\text{MC}}(x) = 1 - p_{(1)}, \qquad a_{\text{M}}(x) = -\big(p_{(1)} - p_{(2)}\big), \qquad
a_{\text{H}}(x) = -\sum_{c} p_c \ln p_c .$$

As escalas, com $K = 10$ classes: $1 - p_{(1)}$ vai de 0 (certeza) a $1 - 1/K = 0{,}9$; a margem vai de 1
(certeza) a 0 (empate perfeito); a entropia (com o logaritmo natural, ln) vai de 0 a $\ln K \approx 2{,}30$ (as 10 igualmente prováveis,
0,1 cada). A margem só olha as duas primeiras; a menor confiança, só a primeira; a entropia, a distribuição
inteira. Primeiro, as três leituras numa imagem só — a do modelo em dúvida, acima:

In [ ]:
PERGUNTA = {"menor confiança": "o favorito é fraco?", "margem": "os dois primeiros empatam?",
            "entropia": "a dúvida se espalha por todos?"}
k = duvida; p = P[k]; i = pool[k]; o = np.argsort(-p)
p1, p2 = p[o[0]], p[o[1]]
ent = -(p * np.log(p + 1e-12)).sum()
conta = {"menor confiança": f"1 − p₁ = 1 − {br(p1, 2)} = {br(1 - p1, 2)}\n(no máximo 0,9)",
         "margem": f"p₁ − p₂ = {br(p1, 2)} − {br(p2, 2)} = {br(p1 - p2, 2)}\n(0 = empate)",
         "entropia": f"−Σ pᵢ ln pᵢ = {br(ent, 2)}\n(no máximo ln 10 = 2,30)"}
fig, eixos = plt.subplots(1, 4, figsize=(14, 4.1), gridspec_kw={"width_ratios": [0.5, 1, 1, 1]})
ax = eixos[0]
ax.imshow(Xp[i].reshape(8, 8), cmap="gray_r"); ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel(f"o oráculo revela:\né um {yp[i]}", fontsize=13)
for j, e in enumerate(["menor confiança", "margem", "entropia"]):
    ax = eixos[j + 1]
    b = ax.bar(modelo.classes_, p, color="#E3E3E3", edgecolor="#9A9A9A")
    if e == "entropia":                          # a entropia olha todas as barras
        for barra in b:
            barra.set_color(TINTA)
        ax.axhline(0.1, color="#555555", ls="--", lw=1.5)
    else:                                        # a menor confiança só p₁; a margem p₁ e p₂
        b[o[0]].set_color(TINTA)
        ax.text(o[0], p1 + 0.012, "p₁", ha="center", va="bottom", fontweight="bold")
        if e == "margem":
            b[o[1]].set_facecolor("white"); b[o[1]].set_edgecolor(TINTA); b[o[1]].set_hatch("///")
            ax.text(o[1], p2 + 0.012, "p₂", ha="center", va="bottom", fontweight="bold")
    ax.set_ylim(0, 0.42); ax.set_xticks(range(10)); ax.set_xlabel("dígito")
    ax.yaxis.set_major_formatter(VIRGULA)
    ax.set_title(f"{e}\n{PERGUNTA[e]}", fontweight="bold")
    ax.text(0.03, 0.96, conta[e], transform=ax.transAxes, va="top", ha="left", fontsize=12.5,
            bbox=dict(facecolor="white", edgecolor="#999999", boxstyle="round,pad=0.3"))
eixos[1].set_ylabel("probabilidade")
fig.tight_layout(w_pad=1.0)
posicao = {e: 1 + int((utilidade(P, e) > utilidade(P, e)[k]).sum()) for e in PERGUNTA}
print(f"esta imagem: é um {yp[i]}, palpite {modelo.classes_[o[0]]} — 1 − p₁ = {1 - p1:.2f} · "
      f"p₁ − p₂ = {p1 - p2:.2f} · entropia = {ent:.2f}")
print("posição desta imagem na fila de cada medida (de " + str(len(pool)) + "): " +
      " · ".join(f"{e} {posicao[e]}ª" for e in PERGUNTA))

Cada medida põe as imagens do pool numa fila diferente — a menor confiança põe esta em primeiro lugar, as
outras não. Perguntamos a cada uma qual imagem ela rotularia agora — cada uma escolhe uma diferente:

In [ ]:
fig, eixos = plt.subplots(2, 3, figsize=(12.5, 6.2), gridspec_kw={"height_ratios": [1.35, 1]})
for j, e in enumerate(["menor confiança", "margem", "entropia"]):
    k = int(np.argmax(utilidade(P, e))); i = pool[k]; p = P[k]; o = np.argsort(-p)
    valor = {"menor confiança": f"1 − p₁ = {br(1 - p[o[0]], 2)}",
             "margem": f"p₁ − p₂ = {br(p[o[0]] - p[o[1]], 3)}",
             "entropia": f"entropia = {br(-(p * np.log(p + 1e-12)).sum(), 2)}"}[e]
    escala = {"menor confiança": "(no máximo 0,9)", "margem": "(0 = empate)",
              "entropia": "(no máximo 2,30: as 10\nem 0,1, a tracejada)"}[e]
    ax = eixos[0, j]
    barras(ax, p, marcar=(e != "entropia"))
    if e == "entropia":
        ax.axhline(0.1, color=TINTA, ls="--", lw=1.5)
    ax.set_ylim(0, 0.42)
    ax.set_title(f"{e}\n{PERGUNTA[e]}", fontweight="bold")
    ax.text(0.03, 0.96, f"{valor}\n{escala}", transform=ax.transAxes, va="top", ha="left", fontsize=12,
            bbox=dict(facecolor="white", edgecolor="#999999", boxstyle="round,pad=0.3"))
    ax = eixos[1, j]
    ax.imshow(Xp[i].reshape(8, 8), cmap="gray_r"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(f"escolhida; o oráculo revela: é um {yp[i]}\n(o palpite do modelo era {modelo.classes_[o[0]]})")
    print(f"{e:16s} escolhe um {yp[i]}: p₁ = {p[o[0]]:.2f} (o {modelo.classes_[o[0]]}), "
          f"p₂ = {p[o[1]]:.2f} (o {modelo.classes_[o[1]]}) — {valor}")
eixos[0, 0].set_ylabel("probabilidade")
fig.tight_layout()

Agora o estudo inteiro: cada estratégia parte de 10 imagens rotuladas ao acaso e ganha uma por passo, até 100;
a acurácia é medida num teste sorteado (30%) que nunca entra no laço. Rodamos tudo com 8 repetições
(sementes): um único sorteio de pontos iniciais engana.

In [ ]:
def curva(estrategia, semente, n0=10, passos=90):
    Xp, Xt, yp, yt = train_test_split(Xd, yd, test_size=0.3, random_state=semente, stratify=yd)
    rng = np.random.default_rng(semente)
    rot = list(rng.choice(len(yp), n0, replace=False))
    acc = []
    for k in range(passos + 1):
        m = LogisticRegression(max_iter=2000).fit(Xp[rot], yp[rot])
        acc.append(m.score(Xt, yt))
        if k == passos:
            break
        pool = np.setdiff1d(np.arange(len(yp)), rot)
        if estrategia == "aleatória":
            rot.append(rng.choice(pool))
        else:
            rot.append(pool[np.argmax(utilidade(m.predict_proba(Xp[pool]), estrategia))])
    return np.array(acc)

ESTRATEGIAS = ["aleatória", "menor confiança", "margem", "entropia"]
curvas = {e: np.array([curva(e, s) for s in range(8)]) for e in ESTRATEGIAS}

In [ ]:
n_rot = 10 + np.arange(91)
fig, ax = plt.subplots(figsize=(10, 4.6))
curvas_com_nome(ax, n_rot, curvas)
ax.set_xlabel("dígitos rotulados"); ax.set_ylabel("acurácia no teste")
ax.set_title("Média de 8 repetições (faixa: a aleatória ± desvio padrão)")

pd.DataFrame({e: {"com 20": A[:, 10].mean(), "com 50": A[:, 40].mean(), "com 100": A[:, 90].mean()}
              for e, A in curvas.items()}).T.round(3)

### Um número para a curva inteira

Comparar só no fim do orçamento esconde o começo, que é onde o aprendizado ativo mais importa. Uma
medida usada nos desafios de aprendizado ativo é a **área sob a curva de aprendizado**: a média da
acurácia ao longo de todo o orçamento.

In [ ]:
for e, A in curvas.items():
    print(f"{e:16s} área sob a curva (acurácia média de 10 a 100 rótulos) = {A.mean(axis=1).mean():.3f} "
          f"± {A.mean(axis=1).std():.3f}")

### Com duas classes, as três medidas são a mesma

Com duas classes, $p_{(1)} = \max(p, 1-p)$ e $p_{(2)} = 1 - p_{(1)}$. Então
$1 - p_{(1)}$, $-(2p_{(1)} - 1)$ e a entropia binária são todas funções **monótonas** de $|p - 0{,}5|$:
ordenam os pontos do mesmo jeito e escolhem o mesmo. As diferenças da tabela acima só existem porque
os dígitos têm 10 classes. Conferindo com mil probabilidades sorteadas:

In [ ]:
p = np.random.default_rng(5).uniform(0, 1, 1000)        # probabilidade da classe 1, mil pontos
P2 = np.c_[1 - p, p]
ordens = {e: np.argsort(-utilidade(P2, e), kind="stable") for e in ["menor confiança", "margem", "entropia"]}
print("mesma ordem de preferência nas três medidas:",
      all(np.array_equal(ordens["menor confiança"], o) for o in ordens.values()))
print("ponto escolhido:", {e: int(o[0]) for e, o in ordens.items()}, f"— p = {p[ordens['margem'][0]]:.4f}")

**Perguntas:**
1. Quantos rótulos a escolha aleatória precisa para chegar à acurácia que a margem tem
   com 50? Leia no gráfico.
2. A entropia quase não ganha da aleatória aqui. Com 10 classes, por que "probabilidade
   espalhada" pode ser uma pista pior do que "dois candidatos empatados"?

## 4. O comitê — e o mesmo laço pronto, no `scikit-activeml`

Na prática não se escreve o laço à mão: o `scikit-activeml` traz dezenas de estratégias.
A convenção dele: os pontos sem rótulo ficam com `MISSING_LABEL` (NaN) no próprio vetor `y`.

Aqui entra uma estratégia nova, o **comitê** (*query by committee*): treinamos vários modelos em
reamostragens dos rótulos e rotulamos onde **eles mais discordam** — não precisa de probabilidade
calibrada.

**A discordância, em fórmula.** Com $C$ membros e $V(c)$ votos para a classe $c$, a **entropia dos
votos** é

$$a_{\text{VE}}(x) = -\sum_{c} \frac{V(c)}{C}\,\log\frac{V(c)}{C}.$$

É zero quando todos concordam e máxima quando os votos se espalham. A ideia vem de Seung, Opper e
Sompolinsky (1992): cada membro é uma hipótese compatível com os rótulos vistos, e onde as hipóteses
discordam está o ponto que mais elimina hipóteses erradas. Aqui o comitê são 5 regressões logísticas
treinadas em reamostragens *bootstrap* dos rótulos (o *query by bagging* de Abe e Mamitsuka, 1998):
`method="vote_entropy"` no `QueryByCommittee`.

**Um exemplo.** As mesmas 30 imagens rotuladas da seção 3; 5 regressões logísticas, cada uma treinada num
sorteio *bootstrap* dessas 30 (com repetição: cada uma vê um conjunto um pouco diferente). Cada uma vota num
dígito para cada imagem do pool:

In [ ]:
comite_ex = BaggingClassifier(LogisticRegression(max_iter=2000), n_estimators=5, random_state=0)
comite_ex.fit(Xp[rot], yp[rot])
votos = np.array([comite_ex.classes_[m.predict(Xp[pool]).astype(int)] for m in comite_ex.estimators_])

def entropia_votos(v):
    f = np.bincount(v, minlength=10) / len(v)
    f = f[f > 0]
    return max(0.0, float(-(f * np.log(f)).sum()))   # sem o -0,0 de quando todos concordam

ev = np.array([entropia_votos(votos[:, j]) for j in range(len(pool))])
concorda = seguro if ev[seguro] == 0 else int(np.flatnonzero(ev == 0)[0])
discorda = int(np.argmax(ev))
fig, eixos = plt.subplots(1, 4, figsize=(13, 3.6), gridspec_kw={"width_ratios": [0.62, 1.5, 0.62, 1.5]})
for j, (k, nome) in enumerate([(concorda, "concordam"), (discorda, "discordam")]):
    i = pool[k]
    ax = eixos[2 * j]
    ax.imshow(Xp[i].reshape(8, 8), cmap="gray_r"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(f"o oráculo revela:\né um {yp[i]}", fontsize=13)
    ax = eixos[2 * j + 1]
    cont = np.bincount(votos[:, k], minlength=10)
    ax.bar(range(10), cont, color=TINTA, edgecolor="#555555")
    ax.set_ylim(0, 5.6); ax.set_yticks(range(6)); ax.set_xticks(range(10)); ax.set_xlabel("dígito")
    ax.set_ylabel("votos (de 5)")
    ax.set_title(f"os 5 {nome}: entropia dos votos = {br(ev[k], 2)}", fontweight="bold")
    print(f"{nome}: votos " + ", ".join(f"{c}×{d}" for d, c in enumerate(cont) if c) +
          f" — entropia dos votos = {ev[k]:.2f} — é um {yp[i]}")
fig.tight_layout(w_pad=1.2)
print(f"máximo possível com 5 votos: ln 5 = {np.log(5):.2f}")

**Conferência.** A margem do `scikit-activeml` escolhe exatamente os mesmos dígitos que a nossa, da seção 3
(mesma divisão, mesmos 10 rótulos iniciais) — por isso o comitê pode ser comparado com as curvas de lá, nas
mesmas 8 repetições.

In [ ]:
from skactiveml.classifier import SklearnClassifier
from skactiveml.pool import QueryByCommittee, RandomSampling, UncertaintySampling
from skactiveml.utils import MISSING_LABEL

CLASSES = np.arange(10)

def curva_skactiveml(nome, semente, n0=10, passos=90):
    Xp, Xt, yp, yt = train_test_split(Xd, yd, test_size=0.3, random_state=semente, stratify=yd)
    y_rot = np.full(len(yp), MISSING_LABEL)
    ini = np.random.default_rng(semente).choice(len(yp), n0, replace=False)
    y_rot[ini] = yp[ini]
    clf = SklearnClassifier(LogisticRegression(max_iter=2000), classes=CLASSES, random_state=semente)
    comite = SklearnClassifier(BaggingClassifier(LogisticRegression(max_iter=2000), n_estimators=5,
                                                 random_state=semente),
                               classes=CLASSES, random_state=semente)
    qs = {"aleatória": RandomSampling(random_state=semente),
          "margem": UncertaintySampling(method="margin_sampling", random_state=semente),
          "comitê": QueryByCommittee(method="vote_entropy", random_state=semente)}[nome]
    acc = []
    for k in range(passos + 1):
        clf.fit(Xp, y_rot)
        acc.append(clf.score(Xt, yt))
        if k == passos:
            break
        if nome == "comitê":
            i = qs.query(X=Xp, y=y_rot, ensemble=comite, fit_ensemble=True)
        elif nome == "margem":
            i = qs.query(X=Xp, y=y_rot, clf=clf, fit_clf=False)
        else:
            i = qs.query(X=Xp, y=y_rot)
        y_rot[i] = yp[i]                           # o "oráculo" revela o rótulo
    return np.array(acc)

for s in (0, 1):
    print(f"repetição {s}: a margem do scikit-activeml dá a mesma curva que a nossa?",
          np.array_equal(curva_skactiveml("margem", s), curvas["margem"][s]))
curvas_comite = np.array([curva_skactiveml("comitê", s) for s in range(8)])

In [ ]:
tres = {"aleatória": curvas["aleatória"], "margem": curvas["margem"], "comitê": curvas_comite}
fig, ax = plt.subplots(figsize=(10, 4.6))
curvas_com_nome(ax, n_rot, tres)
ax.set_xlabel("dígitos rotulados"); ax.set_ylabel("acurácia no teste")
ax.set_title("Média das mesmas 8 repetições (faixa: a aleatória ± desvio padrão)")
for e, A in tres.items():
    print(f"{e:10s} com 20 {A[:, 10].mean():.3f} · com 50 {A[:, 40].mean():.3f} · com 100 {A[:, 90].mean():.3f}")

## 5. Regressão ativa: qual aço medir agora?

É o problema do enigma: cada rótulo é uma
medida (ou uma simulação) cara. Em regressão não há "probabilidade da classe", mas uma
floresta aleatória tem um comitê embutido: **as árvores**. Onde elas discordam — maior
desvio padrão entre as previsões das árvores — o modelo sabe pouco. Chamamos essa estratégia de
**desacordo das árvores**.

Objetivo aqui: um modelo **preciso** com poucas medidas (o erro, RMSE, num conjunto de teste sorteado —
**menor é melhor**). Na Aula 06 o objetivo muda para **achar o máximo** — e a função de aquisição muda junto.

Em fórmula: a floresta prevê a média das $B$ árvores, $\bar T(x) = \frac{1}{B}\sum_b T_b(x)$, e a aquisição é
o desvio padrão entre elas,

$$a(x) = \sqrt{\frac{1}{B}\sum_{b=1}^{B}\big(T_b(x) - \bar T(x)\big)^2} .$$

Não é uma incerteza **calibrada**: as árvores foram treinadas em reamostragens dos mesmos poucos dados
e são correlacionadas entre si, então o desvio subestima o erro real. Para **ordenar** candidatos, que é
o que a aquisição faz, isso basta. Um GP (Aula 05) dá uma incerteza com significado probabilístico,
ao custo de supor uma forma para a função.

Com 312 aços e um teste de 30%, a diferença entre as estratégias é pequena perto do ruído de um sorteio só:
rodamos **20 repetições** e contamos, a cada número de aços medidos, em quantas a escolha ativa errou menos.

In [ ]:
Xa = StandardScaler().fit_transform(acos.drop(columns="limite_escoamento_MPa"))
ya = acos["limite_escoamento_MPa"].to_numpy()

def curva_acos(estrategia, semente, n0=10, passos=60):
    Xp, Xt, yp, yt = train_test_split(Xa, ya, test_size=0.3, random_state=semente)
    rng = np.random.default_rng(semente)
    rot = list(rng.choice(len(yp), n0, replace=False))
    rmse = []
    for k in range(passos + 1):
        f = RandomForestRegressor(n_estimators=50, random_state=0, n_jobs=-1).fit(Xp[rot], yp[rot])
        rmse.append(np.sqrt(np.mean((f.predict(Xt) - yt) ** 2)))
        if k == passos:
            break
        pool = np.setdiff1d(np.arange(len(yp)), rot)
        if estrategia == "aleatória":
            rot.append(rng.choice(pool))
        else:                                     # desacordo das árvores
            arvores = np.stack([a.predict(Xp[pool]) for a in f.estimators_])
            rot.append(pool[np.argmax(arvores.std(axis=0))])
    return np.array(rmse)

curvas_acos = {e: np.array([curva_acos(e, s) for s in range(20)])
               for e in ["aleatória", "desacordo das árvores"]}
n_med = 10 + np.arange(61)
# com 10 aços as duas usam os mesmos 10 (erros iguais): conta só as vitórias de verdade, com folga
vitorias = (curvas_acos["desacordo das árvores"] < curvas_acos["aleatória"] - 1e-6).sum(axis=0)

fig, (ax, bx) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.35, 1]})
curvas_com_nome(ax, n_med, curvas_acos, folga_x=0.62)
ax.set_xlabel("aços medidos"); ax.set_ylabel("RMSE no teste (MPa) — menor é melhor")
ax.set_title("Média de 20 repetições (faixa: a aleatória ± desvio padrão)")
bx.plot(n_med[1:], vitorias[1:], color=LARANJA, lw=2.4, marker="o", markevery=5, ms=6)
bx.axhline(10, color="#7A7A7A", ls="--", lw=1.6)
bx.text(n_med[-1], 0.6, "tracejada: metade (10 de 20)", ha="right", va="bottom", color="#7A7A7A")
bx.set_ylim(0, 20.5); bx.set_yticks([0, 5, 10, 15, 20]); bx.set_xlabel("aços medidos"); bx.grid(alpha=0.3)
bx.set_ylabel("repetições (de 20)")
bx.set_title("Em quantas o desacordo das árvores errou menos")
fig.tight_layout()
for n in (20, 30, 40, 50, 70):
    k = n - 10
    print(f"{n} aços medidos: RMSE aleatória {curvas_acos['aleatória'][:, k].mean():.0f} · "
          f"desacordo {curvas_acos['desacordo das árvores'][:, k].mean():.0f} MPa · "
          f"a escolha ativa errou menos em {vitorias[k]} de 20")
for a, b in ((11, 29), (30, 70)):
    v = vitorias[a - 10:b - 9]
    print(f"de {a} a {b} aços: a escolha ativa errou menos em {v.min()} a {v.max()} das 20 repetições "
          f"(média {v.mean():.1f})")
altas = n_med[vitorias >= 14]
print(f"14 ou mais de 20: entre {altas.min()} e {altas.max()} aços medidos ({len(altas)} dos 61 pontos)")

**Leia os dois painéis juntos.** À esquerda, a média; à direita, a contagem: acima da linha da metade, a
escolha ativa ganha na maioria das repetições. Em que faixa de aços medidos ela ajuda? A partir de quantos
tanto faz? Duas hipóteses (não medidas aqui): no começo, a floresta foi treinada em poucos aços e as árvores,
sorteadas dos mesmos poucos dados, discordam quase ao acaso — o desacordo é uma incerteza ruim (veja acima: não é
calibrada); com muitos aços medidos, qualquer escolha já cobre o espaço — e a vantagem de escolher some. Ela só
aparece no meio.

## 6. Quando o aprendizado ativo atrapalha

### Armadilha 1: medir nos próprios rótulos engana

A estratégia de incerteza escolhe **de propósito** os pontos difíceis, perto da fronteira. O conjunto rotulado
fica cheio deles e já não se parece com os dados de verdade — então uma acurácia estimada **nele** (por
validação cruzada, por exemplo) sai errada. Medimos as duas coisas nas duas luas, com os 24 rótulos de cada
estratégia e 20 repetições: a acurácia estimada por validação cruzada **nos 24 rotulados** (deixa-um-fora) e a
acurácia de verdade **nos 376 pontos que ficaram sem rótulo**.

In [ ]:
medidas = {}
for e in ["aleatória", "incerteza"]:
    cv, fora = [], []
    for s in range(20):
        modelo, rot, _ = laco_2d(e, 20, s)
        rot = np.array(rot)
        cv.append(cross_val_score(SVC(gamma=1.0), X2[rot], y2[rot], cv=LeaveOneOut()).mean())
        resto = np.setdiff1d(np.arange(len(y2)), rot)
        fora.append(modelo.score(X2[resto], y2[resto]))
    medidas[e] = (np.array(cv), np.array(fora))
    print(f"{e:10s} nos 24 rotulados (validação cruzada) {np.mean(cv):.3f} ± {np.std(cv):.3f} · "
          f"nos 376 sem rótulo {np.mean(fora):.3f} ± {np.std(fora):.3f}")

fig, ax = plt.subplots(figsize=(9, 4.6))
x, w = np.arange(2), 0.36
for k, (rotulo, hachura, cor) in enumerate([("medida nos 24 rotulados (validação cruzada)", "///", "white"),
                                            ("medida nos 376 sem rótulo", "", TINTA)]):
    vals = [medidas[e][k].mean() for e in medidas]
    dps = [medidas[e][k].std() for e in medidas]
    ax.bar(x + (k - 0.5) * w, vals, w, yerr=dps, capsize=4, color=cor, hatch=hachura, edgecolor=TINTA,
           label=rotulo)
    for xi, v, d in zip(x + (k - 0.5) * w, vals, dps):
        ax.text(xi, v + d + 0.02, br(v), ha="center", va="bottom", fontweight="bold")
ax.set_xticks(x, ["aleatória", "incerteza"]); ax.set_ylim(0, 1.22); ax.set_ylabel("acurácia")
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.yaxis.set_major_formatter(VIRGULA)
ax.legend(loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.13))
fig.tight_layout()

Na aleatória, as duas medidas concordam. Na incerteza, a validação cruzada nos próprios rótulos diz que o
modelo é ruim, quando ele é o melhor dos dois: os 24 pontos são os mais difíceis que havia.

**Como medir certo: um teste sorteado e guardado antes de começar.** Antes da primeira volta do laço, sorteia-se
uma parte dos dados, pede-se o rótulo deles ao oráculo e eles ficam guardados: nunca entram no treino nem na escolha. A
acurácia (ou o erro) se mede sempre neles. Esses rótulos também custam — e não ensinam nada ao modelo —, mas são
a única medida honesta. Aqui o teste é grande (540 imagens, contra no máximo 100 rótulos no laço) para comparar
as estratégias com precisão; numa campanha real ele é um custo à parte, em geral bem menor. Foi o que fizemos nos dígitos (30% de teste) e nos aços. A divisão dos dígitos:

In [ ]:
_, Xt_, _, yt_ = train_test_split(Xd, yd, test_size=0.3, random_state=0, stratify=yd)
n_t, n_p = len(yt_), len(yd) - len(yt_)
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.add_patch(plt.Rectangle((0, 0), n_p, 1, facecolor="#E6E6E6", edgecolor="#555555", lw=1.5))
ax.add_patch(plt.Rectangle((n_p, 0), n_t, 1, facecolor="white", edgecolor=TINTA, hatch="///", lw=1.5))
ax.text(n_p / 2, 0.5, f"pool: {n_p} imagens\no laço escolhe daqui\n(começa com 10 rotuladas ao acaso)",
        ha="center", va="center", fontsize=13)
ax.text(n_p + n_t / 2, 0.5, f"teste: {n_t} imagens\nsorteadas, rotuladas\ne guardadas",
        ha="center", va="center", fontsize=13, bbox=dict(facecolor="white", edgecolor="none"))
ax.annotate("1. antes do laço: sortear 30% e pedir o rótulo ao oráculo", (n_p + n_t / 2, 1.02), xytext=(n_p + n_t / 2, 1.42),
            ha="center", fontsize=13, arrowprops=dict(arrowstyle="->", color=TINTA, lw=1.8))
ax.text(n_p / 2, -0.2, "2. o laço treina e escolhe só aqui", ha="center", va="top", fontsize=13)
ax.text(n_p + n_t / 2, -0.2, "3. a acurácia é medida só aqui", ha="center", va="top", fontsize=13,
        color=TINTA, fontweight="bold")
ax.set_xlim(-10, len(yd) + 10); ax.set_ylim(-0.55, 1.6); ax.axis("off")
print(f"teste: {n_t} imagens ({n_t / len(yd):.0%}) · pool: {n_p} imagens · total {len(yd)}")

### Armadilha 2: o oráculo também erra

Quem rotula (uma pessoa, um experimento, uma simulação) também erra — e erra mais justamente nos casos
ambíguos, que são os que a incerteza vai buscar. Simulamos um oráculo assim: na fronteira "de verdade" (a de
um SVM treinado com os 400 rótulos) ele acerta no cara ou coroa; longe dela, quase nunca erra — chance de errar
$0{,}5\,e^{-d/0{,}5}$, com $d = |f(x)|$ desse SVM. As duas estratégias rodam de novo, com 24 rótulos e 20
repetições, com o oráculo que nunca erra e com esse.

In [ ]:
mundo = SVC(gamma=1.0).fit(X2, y2)                         # a fronteira "de verdade", com os 400 rótulos
erro = 0.5 * np.exp(-np.abs(mundo.decision_function(X2)) / 0.5)   # a chance de o oráculo errar cada ponto

oraculo = {}
for nome, chance in [("oráculo que nunca erra", None), ("oráculo que erra perto da fronteira", erro)]:
    for e in ["aleatória", "incerteza"]:
        res = [laco_2d(e, 20, s, erro=chance) for s in range(20)]
        acc = np.array([r[0].score(X2, y2) for r in res])
        errados = np.array([sum(r[2][i] != y2[i] for i in r[1]) for r in res])
        oraculo[(nome, e)] = (acc, errados)
        print(f"{nome:36s} {e:10s} acurácia {acc.mean():.3f} ± {acc.std():.3f} · "
              f"rótulos errados recebidos {errados.mean():.1f} (de 20, em média)")
    a, b = oraculo[(nome, "aleatória")][0], oraculo[(nome, "incerteza")][0]
    print(f"{nome:36s} a incerteza ganha em {np.sum(b > a)} de 20 repetições")
for e in ["aleatória", "incerteza"]:
    escolhidos = [i for s in range(20) for i in laco_2d(e, 20, s)[1][4:]]
    print(f"chance média de erro do oráculo nos pontos que a {e} escolhe: {erro[escolhidos].mean():.3f}")

fig, ax = plt.subplots(figsize=(10, 4.6))
rng = np.random.default_rng(0)
for g, nome in enumerate(["oráculo que nunca erra", "oráculo que erra perto da fronteira"]):
    for j, (e, marca) in enumerate([("aleatória", "s"), ("incerteza", "o")]):
        acc, errados = oraculo[(nome, e)]
        xc = 2.4 * g + j
        ax.scatter(xc + rng.uniform(-0.12, 0.12, len(acc)), acc, marker=marca, s=30,
                   color=ESTILO[e]["color"], alpha=0.75)
        ax.plot([xc - 0.28, xc + 0.28], [acc.mean()] * 2, color="black", lw=3)
        ax.text(xc, 0.795, f"{e}\nerrados, em média:\n{br(errados.mean(), 1)} dos 20 rótulos", ha="center",
                va="top", fontsize=11)
    a, b = oraculo[(nome, "aleatória")][0], oraculo[(nome, "incerteza")][0]
    ax.text(2.4 * g + 0.5, 1.005, f"{nome}\na incerteza ganha em {np.sum(b > a)} de 20", ha="center",
            va="bottom", fontweight="bold")
ax.set_xticks([]); ax.set_xlim(-0.6, 3.9); ax.set_ylim(0.70, 1.07); ax.set_yticks(np.arange(0.70, 1.001, 0.05))
ax.set_ylabel("acurácia (cada ponto = uma repetição)"); ax.yaxis.set_major_formatter(VIRGULA)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()

O oráculo ruidoso tira mais da incerteza do que da aleatória: ela vai buscar os pontos onde ele mais erra,
recebe mais rótulos errados, e a vantagem encolhe — e varia mais de uma repetição para outra.

### Outras armadilhas

* **Partida fria.** Com 2 ou 3 rótulos o modelo é ruim, e a incerteza dele também. Comece com alguns pontos
  ao acaso (ou espalhados no espaço). Nos dígitos, com 10 rótulos iniciais o modelo nem viu todos os dígitos.
* **Custo desigual.** Se rotular um ponto custa 10× mais que outro, a aquisição tem de dividir pelo custo.

**O viés de amostragem, formalmente.** O modelo é avaliado sobre a distribuição $p(x)$ dos dados, mas
foi treinado numa amostra escolhida por $a(x)$, que tem outra distribuição. Estimar o erro na amostra
rotulada é como estimar a altura média de um país medindo só o time de basquete. A correção clássica é
pesar cada ponto rotulado pelo inverso da probabilidade de ele ter sido escolhido (*importance
weighting*); a correção prática é a que usamos: um teste sorteado, separado antes do laço.

**Quando parar?** Quando a curva achata — por exemplo, quando a acurácia em validação ou a incerteza
máxima no pool deixam de mudar por alguns rótulos seguidos. Na prática, quase sempre quem decide é o
orçamento.

## 7. Para ir além

* **Redução do erro esperado** (Roy e McCallum, 2001): para cada
  candidato $x$ e cada rótulo possível $y$, retreinar o modelo com $(x, y)$ e medir o erro esperado no
  pool; escolher o $x$ que mais reduz. É a mais direta ao objetivo e a mais cara: um retreino por
  candidato **e** por rótulo.
* **Diversidade em lote.** Rotular 10 pontos de uma vez pelos 10 maiores $a(x)$ costuma escolher 10
  vizinhos quase iguais. Estratégias de lote combinam incerteza com distância entre os escolhidos.
* **Em materiais:** Lookman e colegas (2019) revisam aprendizado ativo e amostragem adaptativa guiados
  por incerteza em ciência de materiais.

### Referências
* Settles, B. *Active Learning Literature Survey*. Computer Sciences Technical Report 1648,
  University of Wisconsin–Madison, 2009 — a revisão clássica, gratuita.
* Lewis, D. D.; Gale, W. A. A sequential algorithm for training text classifiers. *SIGIR*, 1994 — a
  amostragem por incerteza.
* Seung, H. S.; Opper, M.; Sompolinsky, H. Query by committee. *COLT*, 1992.
* Abe, N.; Mamitsuka, H. Query learning strategies using boosting and bagging. *ICML*, 1998.
* Tong, S.; Koller, D. Support vector machine active learning with applications to text
  classification. *Journal of Machine Learning Research*, 2, 2001.
* Roy, N.; McCallum, A. Toward optimal active learning through sampling estimation of error
  reduction. *ICML*, 2001.
* Lookman, T. e colegas. Active learning in materials science with emphasis on adaptive sampling using
  uncertainties for targeted design. *npj Computational Materials*, 5, 2019.
* Documentação do `scikit-activeml`: https://scikit-activeml.github.io/scikit-activeml-docs/

---
# Atividade AL

**Escolha um conjunto de classificação** do scikit-learn que não seja dígitos
(`load_wine`, `load_breast_cancer`, `fetch_covtype` numa amostra…) **ou** um de regressão
de materiais (`data/acos.csv` com outro alvo: o limite de escoamento previsto só com
Cr, Ni, Mo e Co, por exemplo).

1. Separe um **teste ao acaso** (30%) que nunca entra no laço.
2. Compare **três** estratégias, uma delas a aleatória, com o mesmo número inicial de
   rótulos e o mesmo orçamento.
3. Rode **pelo menos 5 repetições** (sementes) e plote a curva de aprendizado com média ± desvio padrão.
4. Responda: quantos rótulos a aleatória precisa para chegar ao desempenho que a melhor
   estratégia tem com metade do orçamento?
5. Mostre um caso em que o aprendizado ativo **não** ajuda (outro modelo, outro número
   inicial, outro conjunto) e explique por quê.

**Cuidados que fazem diferença:**
* O teste é sorteado **antes** do laço e nunca entra nele.
* Uma repetição só é anedota: use várias e mostre média ± desvio padrão.
* A escolha aleatória é sempre a referência — sem ela não há como dizer se a estratégia ajudou.